# Musicm8 — Hybrid DAW / MIDI pipeline

This is now the primary Musicm8 workflow. Instead of trying to learn finished audio from scratch, it:

**songs → Demucs stems → BPM/key → drums/bass/chords/melody MIDI → symbolic arranger → editable MIDI stems → SoundFont preview**

The generated `arrangement.mid` and individual MIDI stems are intended for a real DAW (Ableton, FL Studio, Logic, Reaper, etc.) where you can assign VST instruments, edit notes, automate effects and mix normally.

The first run is slower because Demucs has to split each song. Everything is cached in Google Drive, so later runs reuse the stems and symbolic checkpoint.

> Colab still has to allocate the GPU. If CUDA is unavailable, choose **Runtime → Change runtime type → GPU**, then rerun the same cell.


In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK HYBRID DAW PIPELINE
# songs -> stems -> MIDI -> symbolic model -> DAW bundle
# ============================================================

import os
import sys
import shutil
import subprocess
from pathlib import Path

STEPS = 4000       # symbolic training target; later runs resume automatically
BARS = 8           # generated arrangement length
SEED = 42

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"

AUDIO.mkdir(parents=True, exist_ok=True)
(ROOT / "work").mkdir(parents=True, exist_ok=True)

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)

print("\nInstalling DAW/MIDI dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-daw.txt"], check=True)

print("\nInstalling preview synth...")
subprocess.run(["apt-get", "update", "-qq"], check=True)
installed = False
for packages in (
    ["fluidsynth", "fluid-soundfont-gm"],
    ["fluidsynth", "timgm6mb-soundfont"],
    ["fluidsynth"],
):
    result = subprocess.run(["apt-get", "install", "-y", "-qq", *packages])
    if result.returncode == 0:
        installed = True
        break
if not installed:
    print("⚠️ FluidSynth install failed. MIDI generation can still continue; only the WAV preview may be unavailable.")

import torch
print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU, then rerun THIS SAME CELL.")
print("GPU:", torch.cuda.get_device_name(0))

audio_files = [p for p in AUDIO.rglob("*") if p.suffix.lower() in {".wav",".mp3",".flac",".m4a",".aac",".ogg",".opus"}]
print("Training songs:", len(audio_files))
if not audio_files:
    raise FileNotFoundError(f"Put your songs in {AUDIO}")

cmd = [
    sys.executable, "-u", "daw_workflow.py",
    "--root", str(ROOT),
    "--repo", str(REPO),
    "--steps", str(STEPS),
    "--bars", str(BARS),
    "--seed", str(SEED),
]

print("\n🚀 Starting Musicm8 DAW workflow...")
subprocess.run(cmd, check=True)

PROJECT = ROOT / "work/daw_projects/latest"
PREVIEW = PROJECT / "preview.wav"
ARRANGEMENT = PROJECT / "arrangement.mid"
MIDI_STEMS = PROJECT / "midi_stems"

print("\n✅ EDITABLE DAW OUTPUT")
print("Arrangement:", ARRANGEMENT)
print("MIDI stems :", MIDI_STEMS)
print("Project    :", PROJECT / "project.json")

from IPython.display import Audio, display
if PREVIEW.exists():
    print("\n▶️ SoundFont preview (your VSTs should sound much better):")
    display(Audio(str(PREVIEW)))
else:
    print("\n⚠️ No preview WAV, but the MIDI files are ready.")

print("""
NEXT:
• Import arrangement.mid, or the individual MIDI stems, into your DAW.
• Put a drum VST on drums.mid.
• Put a bass VST on bass.mid.
• Put synth/piano/pad VSTs on chords.mid and melody.mid.
• Edit the MIDI and mix it exactly like a normal production.
""")


## Optional: inspect saved DAW files
Run this only to see what is already cached in Drive.


In [ ]:
from pathlib import Path
root = Path("/content/drive/MyDrive/Musicm8/work")
print("DAW dataset :", root / "daw_dataset")
print("Symbolic ckpt:", root / "daw_symbolic/latest.pt")
print("Latest project:", root / "daw_projects/latest")
for p in sorted((root / "daw_projects/latest").rglob("*")) if (root / "daw_projects/latest").exists() else []:
    if p.is_file():
        print(" -", p)
